In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
from torchvision import models, transforms
from PIL import Image
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from sklearn.metrics import recall_score, precision_score, confusion_matrix
import urllib.parse

# --------------------------------------------------
# 1. 모델 아키텍처
# --------------------------------------------------
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_tabular_features):
        super(MultimodalFusionModel, self).__init__()
        self.vision_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.vision_model.fc = nn.Identity()

        self.tabular_model = nn.Sequential(
            nn.Linear(num_tabular_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, image, tab_data):
        img_features = self.vision_model(image)
        tab_features = self.tabular_model(tab_data)
        combined = torch.cat((img_features, tab_features), dim=1)
        return self.classifier(combined)

# --------------------------------------------------
# 2. 추론기 (Predictor) 클래스
# --------------------------------------------------
class DefectPredictor:
    def __init__(self, model_path, scaler_path, num_features):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = MultimodalFusionModel(num_tabular_features=num_features).to(self.device)
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"모델 가중치 파일 없음: {model_path}")
        
        self.model.load_state_dict(torch.load(model_path, weights_only=True, map_location=self.device))
        self.model.eval()

        if not os.path.exists(scaler_path):
            raise FileNotFoundError(f"스케일러 파일 없음: {scaler_path}")
        self.scaler = joblib.load(scaler_path)

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def predict_batch(self, new_tabular_data, new_image_paths):
        scaled_tab_data = self.scaler.transform(new_tabular_data)
        tab_tensor = torch.tensor(scaled_tab_data, dtype=torch.float32).to(self.device)

        image_tensors = []
        valid_indices = []

        for idx, img_path in enumerate(new_image_paths):
            if not os.path.exists(img_path):
                continue
            
            image = Image.open(img_path).convert('RGB')
            img_tensor = self.transform(image)
            image_tensors.append(img_tensor)
            valid_indices.append(idx)

        if not image_tensors:
            return [], []
        
        image_tensors = torch.stack(image_tensors).to(self.device)
        valid_tab_tensor = tab_tensor[valid_indices]

        predictions = []
        with torch.no_grad():
            outputs = self.model(image_tensors, valid_tab_tensor)
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            predictions = (probs >= 0.5).astype(int)
        
        return predictions, valid_indices

In [ ]:
# 1. DB 연결 (환경에 맞게 수정)
DB_USER = "****"
DB_PASSWORD = "****"
DB_HOST = "****"
DB_PORT = "****"
DB_NAME = "****"

safe_password = urllib.parse.quote_plus(DB_PASSWORD)
db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# 2. 모델/스케일러 로드
model_file = "260402-1156-multi.pt"
scaler_file = "260402-1156-multi_scaler.pkl"

# 3. 테스트 데이터셋 2,000건 로드
query_data = "SELECT * FROM wm_unstructed_datasets"
df_test = pd.read_sql(query_data, engine)
print(f"▶ 테스트 데이터 로드 완료: {len(df_test)}건")

In [ ]:
if df_test.empty:
    print("평가할 데이터가 없습니다.")
else:
    # 1. 정답 라벨 인코딩 (한글 및 영문 혼용 대비)
    df_test = df_test[df_test['ISERROR'].isin(['정상', '불량', 'normal', 'error'])]
    
    # 방어 로직: 필터링 후 데이터가 남아있는지 재확인
    if df_test.empty:
        raise ValueError("필터링 후 평가할 데이터가 0건입니다. DB의 ISERROR 컬럼 값을 확인해주세요.")

    df_test['label'] = df_test['ISERROR'].map({'정상': 0, 'normal': 0, '불량': 1, 'error': 1})
    
    # 2. 치트키 컬럼 및 불필요한 메타데이터 제거
    drop_cols = [
        'PPID', 'LOTID', 'DATIME', 'MODITIME', 'FILENAME', 'FILEPATH', 
        'REMARK', 'ISERROR', 'label', 'WELD_CURR_VAR', 'WELD_CURR_MAX'
    ]
    
    actual_drop_cols = [col for col in drop_cols if col in df_test.columns]
    feature_cols = [col for col in df_test.columns if col not in actual_drop_cols]
    
    X_test_tabular = df_test[feature_cols].select_dtypes(include=[np.number]).values
    test_image_paths = df_test['FILEPATH'].tolist()
    actual_labels = df_test['label'].tolist()
    
    num_features = X_test_tabular.shape[1]

    # 3. 모델 초기화 및 예측 수행
    print("▶ 추론을 시작합니다...")
    predictor = DefectPredictor(model_file, scaler_file, num_features)
    predicted_classes, valid_indices = predictor.predict_batch(X_test_tabular, test_image_paths)
    
    # 4. 결측 이미지로 인해 예측에서 제외된 데이터 필터링
    valid_actual_labels = [actual_labels[i] for i in valid_indices]

    # 5. 성능 지표 계산
    recall = recall_score(valid_actual_labels, predicted_classes, zero_division=0)
    precision = precision_score(valid_actual_labels, predicted_classes, zero_division=0)
    cm = confusion_matrix(valid_actual_labels, predicted_classes)

    print("\n" + "="*40)
    print(" [최종 테스트 평가 결과 (TTA 2,000건)]")
    print("="*40)
    print(f"▶ 평가 데이터 수: {len(valid_indices)}건")
    print(f"▶ RECALL (재현율)  : {recall:.4f}")
    # print("\n[혼동 행렬]")
    # if cm.shape == (2, 2):
    #     print(f" 정답(정상) -> 예측(정상) : {cm[0][0]}")
    #     print(f" 정답(정상) -> 예측(불량) : {cm[0][1]} (과탐지)")
    #     print(f" 정답(불량) -> 예측(정상) : {cm[1][0]} (미탐지) <- Recall 저하 원인")
    #     print(f" 정답(불량) -> 예측(불량) : {cm[1][1]}")
    # else:
    #     print(cm)
    print("="*40)